# Lunar Shadow Detection - Enhanced Preprocessing Pipeline with Metadata
## Project: Shadow-Based Sun Direction Estimation



### **Pipeline Overview**

This notebook implements a complete automated preprocessing pipeline for lunar imagery that:
1. **Automatically converts IMG files to PNG** (planetary image format support)
2. Reads input images with corresponding XML metadata files
3. Tiles long strips into overlapping square segments (256×256 pixels)
4. Generates shadow ground-truth masks automatically
5. Creates metadata files (XML or JSON) for each tile
6. Preserves parent metadata and adds preprocessing information
7. Tracks which tile corresponds to which mask and original image

### **Supported Image Formats**

**Standard**: PNG, JPG, JPEG, TIFF
**Planetary**: IMG (automatically converted to PNG)
   Supports GDAL, raw binary reading, and common lunar image dimensions
   Handles both 8-bit and 16-bit grayscale data




## Step 1: Import Libraries

All libraries are standard Python packages for image processing and XML handling.

In [ ]:

import os
import json
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
from datetime import datetime
import xml.etree.ElementTree as ET
from xml.dom import minidom
from pathlib import Path
import warnings
import struct

# Image processing
from skimage import filters, morphology, exposure
from skimage.util import img_as_ubyte
from scipy import ndimage

# For IMG file reading - install if needed
try:
    from osgeo import gdal
    GDAL_AVAILABLE = True
except ImportError:
    GDAL_AVAILABLE = False
    print(" GDAL not available - will use basic IMG reading")

#We will  Allow loading large images
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

#Good to suppress warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully!")
print(f" Pipeline initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

⚠ GDAL not available - will use basic IMG reading
✓ Libraries imported successfully!
✓ Pipeline initialized at: 2026-02-08 01:16:17


## Step 2: Configuration Parameters

**Theory Reference**: Section 2.2 (Optimal Tile Size Selection)

Adjust these parameters based on your data:

In [ ]:
# =====================================
# CONFIG PARAMETERS
# ==================================

# Directory paths
INPUT_IMG_DIR = "input_images"           #  image files here
INPUT_METADATA_DIR = "input_metadata"    # Placed corresponding XML files here
OUTPUT_PNG_DIR = "output_png"            # Converted PNG files (if from IMG)
OUTPUT_TILES_DIR = "output_tiles"        # 256×256 tiles
OUTPUT_MASKS_DIR = "output_masks"        # Shadow masks
OUTPUT_METADATA_DIR = "output_metadata"  #  JSON metadata or even XML 
OUTPUT_VIZ_DIR = "output_visualizations" 
# Metadata format ('xml' or 'json')
OUTPUT_METADATA_FORMAT = 'json'  # 'xml' if needed

# Tiling parameters
TILE_SIZE = 256          #  256 pixels (optimal balance)
OVERLAP_FRACTION = 0.5   # 50% overlap (preserves shadow continuity)
STRIDE = int(TILE_SIZE * (1 - OVERLAP_FRACTION))  # = 128 pixels

# shadow detection parameters 
PERCENTILE_LOW = 2       # Removing dark outliers
PERCENTILE_HIGH = 98     # Remove bright outliers
MORPH_KERNEL_SIZE = 3    

MIN_TILE_INTENSITY = 0.01  # Skip nearly black tiles
MAX_TILE_INTENSITY = 0.99  # Skip nearly white tiles

# Create output directories
for directory in [INPUT_METADATA_DIR, OUTPUT_PNG_DIR, OUTPUT_TILES_DIR, 
                  OUTPUT_MASKS_DIR, OUTPUT_METADATA_DIR, OUTPUT_VIZ_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f" Configuration loaded:")
print(f"  - Tile size: {TILE_SIZE}×{TILE_SIZE} pixels")
print(f"  - Overlap: {OVERLAP_FRACTION*100}% (stride = {STRIDE} pixels)")
print(f"  - Output metadata format: {OUTPUT_METADATA_FORMAT.upper()}")
print(f"  - Output directories created")

# Calculate expected number of tiles
def estimate_tile_count(image_height, image_width):
    """Estimates number of tiles from image dimensions."""
    ny = (image_height - TILE_SIZE) // STRIDE + 1
    nx = (image_width - TILE_SIZE) // STRIDE + 1
    return ny * nx

# Example: 10000×5000 pixel image
example_tiles = estimate_tile_count(10000, 5000)
print(f"  - Example: 10000×5000 image → ~{example_tiles} tiles")

✓ Configuration loaded:
  - Tile size: 256×256 pixels
  - Overlap: 50.0% (stride = 128 pixels)
  - Output metadata format: JSON
  - Output directories created
  - Example: 10000×5000 image → ~2926 tiles


## Step 3: XML Metadata Utilities

Functions to parse, manipulate, and save XML metadata files.

In [ ]:
def parse_xml_metadata(xml_filepath):
    """
    Parses XML metadata file into a dictionary.
    
    Args:
        xml_filepath: Path to XML file
    
    Returns:
        Dictionary containing all XML elements and attributes
    """
    if not os.path.exists(xml_filepath):
        print(f" Warning: XML file not found: {xml_filepath}")
        return {}
    
    try:
        tree = ET.parse(xml_filepath)
        root = tree.getroot()
        
        def element_to_dict(element):
            """Recursively converts XML element to dictionary."""
            result = {}
            
            # Add attributes
            if element.attrib:
                result['@attributes'] = element.attrib
            
            # Add text content
            if element.text and element.text.strip():
                result['#text'] = element.text.strip()
            
            # Add children
            for child in element:
                child_data = element_to_dict(child)
                
                if child.tag in result:
                    # Handle multiple children with same tag
                    if not isinstance(result[child.tag], list):
                        result[child.tag] = [result[child.tag]]
                    result[child.tag].append(child_data)
                else:
                    result[child.tag] = child_data
            
            return result
        
        metadata = {
            'root_tag': root.tag,
            'data': element_to_dict(root)
        }
        
        return metadata
    
    except Exception as e:
        print(f" Error parsing XML {xml_filepath}: {e}")
        return {}


def dict_to_xml_element(tag, data, parent=None):
    """
    Converts dictionary to XML element.
    
    Args:
        tag: XML tag name
        data: Dictionary data
        parent: Parent XML element (if any)
    
    Returns:
        XML element
    """
    if parent is not None:
        element = ET.SubElement(parent, tag)
    else:
        element = ET.Element(tag)
    
    if isinstance(data, dict):
        # Add attributes
        if '@attributes' in data:
            for key, value in data['@attributes'].items():
                element.set(key, str(value))
        
        # Add text content
        if '#text' in data:
            element.text = str(data['#text'])
        
        # Add children
        for key, value in data.items():
            if key not in ['@attributes', '#text']:
                if isinstance(value, list):
                    for item in value:
                        dict_to_xml_element(key, item, element)
                else:
                    dict_to_xml_element(key, value, element)
    else:
        # Simple value
        element.text = str(data)
    
    return element


def save_metadata_xml(metadata_dict, output_filepath):
    """
    Saves metadata dictionary as formatted XML file.
    
    Args:
        metadata_dict: Dictionary containing metadata
        output_filepath: Path to save XML file
    """
    # Create root element
    root_tag = metadata_dict.get('root_tag', 'metadata')
    root_data = metadata_dict.get('data', metadata_dict)
    
    root = dict_to_xml_element(root_tag, root_data)
    
    # Convert to pretty-printed string
    rough_string = ET.tostring(root, encoding='unicode')
    reparsed = minidom.parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")
    
    # Remove extra blank lines
    lines = [line for line in pretty_xml.split('\n') if line.strip()]
    pretty_xml = '\n'.join(lines)
    
    # Save to file
    with open(output_filepath, 'w', encoding='utf-8') as f:
        f.write(pretty_xml)


def save_metadata_json(metadata_dict, output_filepath):
    """
    Saves metadata dictionary as formatted JSON file.
    
    Args:
        metadata_dict: Dictionary containing metadata
        output_filepath: Path to save JSON file
    """
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(metadata_dict, f, indent=2, ensure_ascii=False)


def get_metadata_filepath(image_filepath, metadata_dir=INPUT_METADATA_DIR):
    """
    Gets the corresponding metadata filepath for an image.
    
    Args:
        image_filepath: Path to image file
        metadata_dir: Directory containing metadata files
    
    Returns:
        Path to metadata file (XML)
    """
    image_name = Path(image_filepath).stem
    xml_filepath = os.path.join(metadata_dir, f"{image_name}.xml")
    return xml_filepath


print(" XML metadata utilities defined")
print("\nFunctions available:")
print("  - parse_xml_metadata(): Parse XML file to dictionary")
print("  - save_metadata_xml(): Save dictionary as XML")
print("  - save_metadata_json(): Save dictionary as JSON")
print("  - get_metadata_filepath(): Get metadata path for image")

✓ XML metadata utilities defined

Functions available:
  - parse_xml_metadata(): Parse XML file to dictionary
  - save_metadata_xml(): Save dictionary as XML
  - save_metadata_json(): Save dictionary as JSON
  - get_metadata_filepath(): Get metadata path for image


## Step 4: IMG File Conversion (planetary Image)

Functions to convert IMG format files to standard PNG format.

In [ ]:
def normalize_image_global(image_array, percentile_low=2, percentile_high=98):
    """
    Applies global normalization to image.
    
    Theory: Section 4.2 - Preserves relative brightness ratios
    
    Args:
        image_array: numpy array of image
        percentile_low: lower percentile for clipping
        percentile_high: upper percentile for clipping
    
    Returns:
        normalized image array [0, 1]
    """
    # Compute percentiles on entire image
    p_low = np.percentile(image_array, percentile_low)
    p_high = np.percentile(image_array, percentile_high)
    
    # Clip and normalize
    clipped = np.clip(image_array, p_low, p_high)
    
    if p_high > p_low:
        normalized = (clipped - p_low) / (p_high - p_low)
    else:
        normalized = clipped * 0  # All same value
    
    return normalized.astype(np.float32)


def generate_shadow_mask(tile, morph_kernel_size=3):
    """
    Generates binary shadow mask using Otsu's thresholding.
    
    Theory: Section 5.2 - Physics-based shadow detection
    
    Args:
        tile: normalized tile image [0, 1]
        morph_kernel_size: kernel size for morphological operations
    
    Returns:
        binary mask (1 = shadow, 0 = sunlit)
    """
    # Otsu's thresholding
    threshold = filters.threshold_otsu(tile)
    mask = tile < threshold  # Shadows are darker
    
    # Morphological cleaning
    kernel = morphology.disk(morph_kernel_size)
    
    # Remove small noise (opening)
    mask = morphology.binary_opening(mask, kernel)
    
    # Fill small holes (closing)
    mask = morphology.binary_closing(mask, kernel)
    
    return mask.astype(np.uint8)


def compute_shadow_statistics(mask):
    """
    Computes statistics about shadow mask.
    
    Args:
        mask: binary shadow mask
    
    Returns:
        dictionary of shadow statistics
    """
    total_pixels = mask.size
    shadow_pixels = np.sum(mask)
    shadow_fraction = shadow_pixels / total_pixels if total_pixels > 0 else 0
    
    # Compute connected components
    labeled_mask, num_components = ndimage.label(mask)
    
    stats = {
        'shadow_pixel_count': int(shadow_pixels),
        'total_pixel_count': int(total_pixels),
        'shadow_fraction': float(shadow_fraction),
        'num_shadow_regions': int(num_components)
    }
    
    return stats


print(" Preprocessing functions defined")
print("\nFunctions available:")
print("  - normalize_image_global(): Global normalization")
print("  - generate_shadow_mask(): Otsu + morphology")
print("  - compute_shadow_statistics(): Shadow metrics")

✓ Preprocessing functions defined

Functions available:
  - normalize_image_global(): Global normalization
  - generate_shadow_mask(): Otsu + morphology
  - compute_shadow_statistics(): Shadow metrics


## Step 5: Image Tiling with Metadata Generation

Main pipeline function that tiles images and generates metadata for each tile.

In [61]:
##
## Image Format Conversion Functions

#Functions to convert planetary IMG files to standard PNG format.


In [ ]:
def convert_img_to_png(img_filepath, output_dir=OUTPUT_PNG_DIR):
    """
    Converts NASA/ISRO IMG format files to PNG.
    
    Supports:
    1. GDAL (if available)
    2. XML metadata extraction
    3. Raw binary reading with auto-detection
    
    Args:
        img_filepath: Path to .img file
        output_dir: Directory to save converted PNG
    
    Returns:
        Path to converted PNG file, or None if conversion failed
    """
    img_filename = Path(img_filepath).stem
    output_png = os.path.join(output_dir, f"{img_filename}.png")
    
    # If already converted, return existing file
    if os.path.exists(output_png):
        print(f"   Using existing PNG: {os.path.basename(output_png)}")
        return output_png
    
    try:
        # Method 1: Try XML extraction first
        print(f"  → Attempting XML extraction...")
        img_name = Path(img_filepath).stem
        xml_filepath = get_metadata_filepath(img_filepath)
        
        if os.path.exists(xml_filepath):
            dims = extract_dimensions_from_xml(xml_filepath)
            if dims:
                width, height = dims
                file_size = os.path.getsize(img_filepath)
                
                # Try conversion with XML dimensions
                try:
                    with open(img_filepath, 'rb') as f:
                        raw_data = f.read()
                    
                    # Try 16-bit first (most common for lunar imagery)
                    if width * height * 2 == file_size:
                        print(f"  → Converting as {width}×{height} (16-bit from XML)...")
                        image_array = np.frombuffer(raw_data, dtype=np.uint16).reshape((height, width))
                        normalized = (image_array.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
                        Image.fromarray(normalized).save(output_png)
                        print(f"  ✓ Converted to: {os.path.basename(output_png)}")
                        return output_png
                    
                    # Try 8-bit
                    elif width * height == file_size:
                        print(f"  → Converting as {width}×{height} (8-bit from XML)...")
                        image_array = np.frombuffer(raw_data, dtype=np.uint8).reshape((height, width))
                        Image.fromarray(image_array).save(output_png)
                        print(f"  ✓ Converted to: {os.path.basename(output_png)}")
                        return output_png
                    
                except Exception as e:
                    print(f"  ✗ XML dimensions didn't match: {e}")
        
        # Method 2: Try GDAL
        if GDAL_AVAILABLE:
            print(f"   Converting with GDAL------------")
            dataset = gdal.Open(img_filepath)
            
            if dataset is not None:
                band = dataset.GetRasterBand(1)
                image_array = band.ReadAsArray()
                
                if image_array is not None:
                    img_min = np.min(image_array)
                    img_max = np.max(image_array)
                    
                    if img_max > img_min:
                        normalized = ((image_array - img_min) / (img_max - img_min) * 255).astype(np.uint8)
                    else:
                        normalized = image_array.astype(np.uint8)
                    
                    Image.fromarray(normalized).save(output_png)
                    print(f"   Converted to: {os.path.basename(output_png)}")
                    return output_png
        
        # Method 3: Try raw binary reading
        print(f"  → Converting with binary reading...")
        return read_raw_img_file(img_filepath, output_png)
    
    except Exception as e:
        print(f" -------- Conversion failed: {e}")
        return None


def read_raw_img_file(img_filepath, output_png_path):
    """
    Reads raw binary IMG file and converts to PNG.
    
    Works for simple rectangular image files.
    Auto-detects dimensions from file size and XML metadata.
    
    Args:
        img_filepath: Path to .img file
        output_png_path: Path to save PNG
    
    Returns:
        Path to PNG if successful, None otherwise
    """
    try:
        # Read raw binary data
        with open(img_filepath, 'rb') as f:
            raw_data = f.read()
        
        file_size_bytes = len(raw_data)
        print(f"  File size: {file_size_bytes} bytes")
        
        # Step 1: Try to get dimensions from XML metadata
        img_name = Path(img_filepath).stem
        xml_path = get_metadata_filepath(img_filepath)
        
        xml_dims = None
        if os.path.exists(xml_path):
            try:
                xml_metadata = parse_xml_metadata(xml_path)
                # Try to extract dimensions from XML
                data = xml_metadata.get('data', {})
                
                # Common XML tags for image dimensions
                for key in ['WIDTH', 'width', 'Width', 'LINES', 'lines', 'SAMPLES', 'samples']:
                    if key in data:
                        dimension_value = data[key]
                        if isinstance(dimension_value, dict):
                            dimension_value = dimension_value.get('#text', dimension_value)
                        try:
                            print(f"    Found {key}: {dimension_value}")
                        except:
                            pass
            except:
                pass
        
        # Step 2: Try different bit depths and look for rectangular dimensions
        for bits_per_pixel in [16, 8]:
            bytes_per_pixel = bits_per_pixel // 8
            pixels_available = file_size_bytes // bytes_per_pixel
            
            # Try square dimensions first
            side = int(np.sqrt(pixels_available))
            
            if side * side * bytes_per_pixel == file_size_bytes:
                # Perfect square fit!
                if bits_per_pixel == 16:
                    image_array = np.frombuffer(raw_data, dtype=np.uint16).reshape((side, side))
                else:
                    image_array = np.frombuffer(raw_data, dtype=np.uint8).reshape((side, side))
                
                # Normalize to 0-255
                if bits_per_pixel == 16:
                    normalized = (image_array.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
                else:
                    normalized = image_array
                
                Image.fromarray(normalized).save(output_png_path)
                print(f"  ✓ Converted {side}×{side} image to: {os.path.basename(output_png_path)}")
                return output_png_path
        
        # Step 3: Try common square dimensions
        common_square_dims = [
            5000, 4096, 3000, 2532, 2048, 1024, 8192, 10000, 11520, 12000
        ]
        
        for dim in common_square_dims:
            expected_size_16bit = dim * dim * 2
            expected_size_8bit = dim * dim
            
            if file_size_bytes == expected_size_16bit:
                image_array = np.frombuffer(raw_data, dtype=np.uint16).reshape((dim, dim))
                normalized = (image_array.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
                Image.fromarray(normalized).save(output_png_path)
                print(f"  ✓ Converted {dim}×{dim} image to: {os.path.basename(output_png_path)}")
                return output_png_path
            
            elif file_size_bytes == expected_size_8bit:
                image_array = np.frombuffer(raw_data, dtype=np.uint8).reshape((dim, dim))
                Image.fromarray(image_array).save(output_png_path)
                print(f"  ✓ Converted {dim}×{dim} image to: {os.path.basename(output_png_path)}")
                return output_png_path
        
        # Step 4: Try common rectangular dimensions
        common_rect_dims = [
            (5000, 5000), (2532, 2304), (3000, 3000), (4096, 4096), (8192, 8192),
            (11520, 10260), (12000, 10000), (23040, 5740), (46080, 2870),  # LROC images
            (10240, 10240), (6144, 5120), (16384, 8192)  # Other common sizes
        ]
        
        for width, height in common_rect_dims:
            expected_size_16bit = width * height * 2
            expected_size_8bit = width * height
            
            if file_size_bytes == expected_size_16bit:
                image_array = np.frombuffer(raw_data, dtype=np.uint16).reshape((height, width))
                normalized = (image_array.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
                Image.fromarray(normalized).save(output_png_path)
                print(f"   Converted {width}×{height} image to: {os.path.basename(output_png_path)}")
                return output_png_path
            
            elif file_size_bytes == expected_size_8bit:
                image_array = np.frombuffer(raw_data, dtype=np.uint8).reshape((height, width))
                Image.fromarray(image_array).save(output_png_path)
                print(f"   Converted {width}×{height} image to: {os.path.basename(output_png_path)}")
                return output_png_path
        
        # Step 5: Try to find factors (for non-standard dimensions)
        print(f"  → Trying to find image dimensions from file size...")
        for bits_per_pixel in [16, 8]:
            bytes_per_pixel = bits_per_pixel // 8
            
            if file_size_bytes % bytes_per_pixel != 0:
                continue
            
            pixels = file_size_bytes // bytes_per_pixel
            
            # Try to find a reasonable width/height pair
            # Assume width is typically between 1000 and 50000
            for width in range(1000, min(50001, pixels // 2), 100):
                if pixels % width == 0:
                    height = pixels // width
                    
                    # Check if it's a reasonable dimension
                    if 1000 <= height <= 50000 and abs(width - height) < 50000:
                        try:
                            if bits_per_pixel == 16:
                                image_array = np.frombuffer(raw_data, dtype=np.uint16).reshape((height, width))
                                normalized = (image_array.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
                            else:
                                image_array = np.frombuffer(raw_data, dtype=np.uint8).reshape((height, width))
                                normalized = image_array
                            
                            Image.fromarray(normalized).save(output_png_path)
                            print(f"  ✓ Converted {width}×{height} image ({bits_per_pixel}-bit) to: {os.path.basename(output_png_path)}")
                            return output_png_path
                        except Exception as e:
                            continue
        
        print(f"   File size: {file_size_bytes} bytes")
        print(f"   If 16-bit: {file_size_bytes // 2} pixels")
        print(f"  If 8-bit: {file_size_bytes} pixels")
        print(f"   Could not auto-detect dimensions. Please specify manually or check XML metadata.")
        return None
    
    except Exception as e:
        print(f"   Raw image reading failed: {e}")
        return None


print(" IMG file conversion functions defined")
print("\nFunctions available:")
print("  - convert_img_to_png(): Convert IMG to PNG")
print("  - read_raw_img_file(): Read raw binary IMG files")

✓ IMG file conversion functions defined

Functions available:
  - convert_img_to_png(): Convert IMG to PNG
  - read_raw_img_file(): Read raw binary IMG files


In [ ]:
def process_image_with_metadata(image_filepath, 
                                 tile_size=TILE_SIZE,
                                 stride=STRIDE,
                                 generate_masks=True,
                                 save_visualizations=True):
    """
    Processes a single image: tiles it, generates masks, and creates metadata.
    
    Args:
        image_filepath: Path to input image
        tile_size: Size of square tiles
        stride: Stride for sliding window
        generate_masks: Whether to generate shadow masks
        save_visualizations: Whether to save visualization images
    
    Returns:
        Number of tiles processed
    """
    print(f"\nProcessing: {os.path.basename(image_filepath)}")
    
    # Load image
    image = Image.open(image_filepath)
    image_array = np.array(image)
    
    # Convert to grayscale if needed
    if len(image_array.shape) == 3:
        image_array = np.mean(image_array, axis=2)
    
    height, width = image_array.shape
    image_name = Path(image_filepath).stem
    
    print(f"  Image size: {width}×{height} pixels")
    
    # Load parent metadata
    parent_metadata_filepath = get_metadata_filepath(image_filepath)
    parent_metadata = parse_xml_metadata(parent_metadata_filepath)
    
    if parent_metadata:
        print(f"   Loaded parent metadata from: {os.path.basename(parent_metadata_filepath)}")
    else:
        print(f"   No parent metadata found, using empty metadata")
        parent_metadata = {'root_tag': 'metadata', 'data': {}}
    
    # Global normalization
    normalized_image = normalize_image_global(
        image_array, 
        PERCENTILE_LOW, 
        PERCENTILE_HIGH
    )
    
    # Calculate tile positions
    tile_count = 0
    tiles_processed = []
    
    for y in range(0, height - tile_size + 1, stride):
        for x in range(0, width - tile_size + 1, stride):
            # Extract tile
            tile = normalized_image[y:y+tile_size, x:x+tile_size]
            
            # Quality check
            mean_intensity = np.mean(tile)
            if mean_intensity < MIN_TILE_INTENSITY or mean_intensity > MAX_TILE_INTENSITY:
                continue  # Skip nearly black or white tiles
            
            # Generate tile ID
            tile_id = f"{image_name}_tile_{tile_count:04d}"
            
            # Save tile image
            tile_filepath = os.path.join(OUTPUT_TILES_DIR, f"{tile_id}.png")
            tile_uint8 = img_as_ubyte(tile)
            Image.fromarray(tile_uint8).save(tile_filepath)
            
            # Generate shadow mask
            mask = None
            mask_filepath = None
            shadow_stats = None
            
            if generate_masks:
                mask = generate_shadow_mask(tile, MORPH_KERNEL_SIZE)
                shadow_stats = compute_shadow_statistics(mask)
                
                # Save mask
                mask_filepath = os.path.join(OUTPUT_MASKS_DIR, f"{tile_id}_mask.png")
                Image.fromarray(mask * 255).save(mask_filepath)
            
            # Create tile metadata
            tile_metadata = create_tile_metadata(
                parent_metadata=parent_metadata,
                tile_id=tile_id,
                parent_image_name=image_name,
                tile_position=(x, y),
                tile_size=tile_size,
                parent_image_size=(width, height),
                mean_intensity=float(mean_intensity),
                shadow_stats=shadow_stats,
                mask_filename=os.path.basename(mask_filepath) if mask_filepath else None
            )
            
            # Save metadata
            metadata_ext = OUTPUT_METADATA_FORMAT
            metadata_filepath = os.path.join(OUTPUT_METADATA_DIR, f"{tile_id}.{metadata_ext}")
            
            if OUTPUT_METADATA_FORMAT == 'xml':
                save_metadata_xml(tile_metadata, metadata_filepath)
            else:
                save_metadata_json(tile_metadata, metadata_filepath)
            
            # Save visualization
            if save_visualizations and mask is not None:
                viz_filepath = os.path.join(OUTPUT_VIZ_DIR, f"{tile_id}_viz.png")
                save_visualization(tile, mask, viz_filepath)
            
            tiles_processed.append({
                'tile_id': tile_id,
                'position': (x, y),
                'shadow_fraction': shadow_stats['shadow_fraction'] if shadow_stats else 0
            })
            
            tile_count += 1
    
    print(f"   Generated {tile_count} tiles")
    print(f"   Saved to: {OUTPUT_TILES_DIR}/")
    print(f"   Masks saved to: {OUTPUT_MASKS_DIR}/")
    print(f"   Metadata saved to: {OUTPUT_METADATA_DIR}/")
    
    return tile_count


def create_tile_metadata(parent_metadata, 
                         tile_id,
                         parent_image_name,
                         tile_position,
                         tile_size,
                         parent_image_size,
                         mean_intensity,
                         shadow_stats=None,
                         mask_filename=None):
    """
    Creates metadata for a single tile by combining parent metadata with preprocessing info.
    
    Args:
        parent_metadata: Dictionary of parent image metadata
        tile_id: Unique tile identifier
        parent_image_name: Name of parent image
        tile_position: (x, y) position in parent image
        tile_size: Size of tile in pixels
        parent_image_size: (width, height) of parent image
        mean_intensity: Mean pixel intensity of tile
        shadow_stats: Dictionary of shadow statistics
        mask_filename: Filename of corresponding mask
    
    Returns:
        Dictionary containing complete tile metadata
    """
    # Start with parent metadata (this preserves all parent fields)
    tile_metadata = {
        'root_tag': parent_metadata.get('root_tag', 'tile_metadata'),
        'data': parent_metadata.get('data', {}).copy() if isinstance(parent_metadata.get('data'), dict) else {}
    }
    
    # Add preprocessing information (unique to this tile)
    preprocessing_info = {
        'tile_info': {
            'tile_id': tile_id,
            'parent_image': parent_image_name,
            'tile_filename': f"{tile_id}.png",
            'mask_filename': mask_filename if mask_filename else 'N/A'
        },
        'spatial_info': {
            'tile_size_pixels': tile_size,
            'position_x': tile_position[0],
            'position_y': tile_position[1],
            'parent_image_width': parent_image_size[0],
            'parent_image_height': parent_image_size[1]
        },
        'preprocessing': {
            'normalization_method': 'global_percentile',
            'percentile_low': PERCENTILE_LOW,
            'percentile_high': PERCENTILE_HIGH,
            'mean_intensity': mean_intensity,
            'tile_overlap_fraction': OVERLAP_FRACTION,
            'stride_pixels': STRIDE
        },
        'processing_timestamp': datetime.now().isoformat()
    }
    
    # Add shadow statistics if available
    if shadow_stats:
        preprocessing_info['shadow_detection'] = {
            'method': 'otsu_thresholding',
            'morphology_kernel_size': MORPH_KERNEL_SIZE,
            **shadow_stats
        }
    
    # Merge preprocessing info into tile metadata
    tile_metadata['data']['preprocessing_pipeline'] = preprocessing_info
    
    return tile_metadata


def save_visualization(tile, mask, output_filepath):
    """
    Creates and saves a visualization of tile with mask overlay.
    
    Args:
        tile: Normalized tile image
        mask: Binary shadow mask
        output_filepath: Path to save visualization
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original tile
    axes[0].imshow(tile, cmap='gray')
    axes[0].set_title('Tile')
    axes[0].axis('off')
    
    # Shadow mask
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Shadow Mask')
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(tile, cmap='gray')
    shadow_overlay = np.zeros((*mask.shape, 4))
    shadow_overlay[mask == 1] = [1, 0, 0, 0.4]  # Red with 40% opacity
    axes[2].imshow(shadow_overlay)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_filepath, dpi=100, bbox_inches='tight')
    plt.close()


print("Image tiling and metadata functions defined")
print("\nMain function: process_image_with_metadata()")

✓ Image tiling and metadata functions defined

Main function: process_image_with_metadata()


## Step 6: Batch Processing

Process all images in the input directory.

In [ ]:
def process_all_images():
    """
    Processes all images in the input directory.
    
    Handles both standard formats (PNG, JPG) and planetary formats (IMG).
    Automatically converts IMG files to PNG before processing.
    
    Returns:
        Summary statistics
    """
    # Find all image files
    image_extensions = ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.img']
    image_files = []
    
    for ext in image_extensions:
        image_files.extend(Path(INPUT_IMG_DIR).glob(f'*{ext}'))
        image_files.extend(Path(INPUT_IMG_DIR).glob(f'*{ext.upper()}'))
    
    if not image_files:
        print(f" No images found in {INPUT_IMG_DIR}/")
        print(f"  Supported formats: {', '.join(image_extensions)}")
        return
    
    print(f"\n{'='*60}")
    print(f"BATCH PROCESSING: {len(image_files)} images found")
    print(f"{'='*60}")
    
    total_tiles = 0
    successful_images = 0
    failed_images = []
    converted_files = []
    
    # Step 1: Convert IMG files to PNG
    print(f"\n{'='*60}")
    print("STEP 1: Converting IMG files to PNG")
    print(f"{'='*60}")
    
    for image_file in image_files:
        if image_file.suffix.lower() == '.img':
            print(f"\nConverting: {image_file.name}")
            converted_path = convert_img_to_png(str(image_file))
            
            if converted_path:
                converted_files.append(Path(converted_path))
            else:
                failed_images.append(image_file.name)
    
    # Use converted files if any IMG files were found, otherwise use originals
    if converted_files:
        files_to_process = converted_files
        print(f"\n Converted {len(converted_files)} IMG file(s)")
    else:
        # Filter out IMG files if conversion failed
        files_to_process = [f for f in image_files if f.suffix.lower() != '.img']
    
    # Step 2: Process all images
    print(f"\n{'='*60}")
    print(f"STEP 2: Processing {len(files_to_process)} images")
    print(f"{'='*60}")
    
    for i, image_file in enumerate(files_to_process, 1):
        try:
            print(f"\n[{i}/{len(files_to_process)}] Processing: {image_file.name}")
            tiles = process_image_with_metadata(str(image_file))
            total_tiles += tiles
            successful_images += 1
        except Exception as e:
            print(f"   Error processing {image_file.name}: {e}")
            failed_images.append(image_file.name)
    
    # Print summary
    print(f"\n{'='*60}")
    print("PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Successfully processed: {successful_images}/{len(files_to_process)} images")
    print(f" Total tiles generated: {total_tiles}")
    print(f" Metadata format: {OUTPUT_METADATA_FORMAT.upper()}")
    
    if failed_images:
        print(f"\n Failed images ({len(failed_images)}):")
        for name in failed_images:
            print(f"  - {name}")
    
    print(f"\nOutput locations:")
    print(f"  - Tiles: {OUTPUT_TILES_DIR}/")
    print(f"  - Masks: {OUTPUT_MASKS_DIR}/")
    print(f"  - Metadata: {OUTPUT_METADATA_DIR}/")
    print(f"  - Visualizations: {OUTPUT_VIZ_DIR}/")
    print(f"  - Converted PNG: {OUTPUT_PNG_DIR}/")

## Step 7: Run the Pipeline

Execute the complete preprocessing pipeline.

In [ ]:
# UNCOMMENT & RUN THE PIPELINE
# process_all_images()

## Step 8: Process Single Image (Example)

Process a single image for testing.

In [ ]:
#  Process a single image for example
# Replace 'your_image.png' with your actual image filename

# example_image = os.path.join(INPUT_IMG_DIR, 'your_image.png')
# if os.path.exists(example_image):
#     tiles_count = process_image_with_metadata(example_image)
#     print(f"\n✓ Generated {tiles_count} tiles from example image")
# else:
#     print(f"⚠ Example image not found: {example_image}")

In [67]:
## Helper: Analyze IMG File Dimensions

#Use this to determine the exact dimensions of your IMG files.


In [ ]:
def analyze_img_dimensions(img_filepath):
    """
    Analyzes IMG file to determine possible dimensions.
    
    Tests all possible width/height combinations and suggests the most likely ones.
    Prints both 16-bit and 8-bit possibilities.
    
    Args:
        img_filepath: Path to .img file
    """
    file_size = os.path.getsize(img_filepath)
    filename = os.path.basename(img_filepath)
    
    print(f"\n{'='*70}")
    print(f"Analyzing: {filename}")
    print(f"{'='*70}")
    print(f"File size: {file_size:,} bytes\n")
    
    results = []
    
    # Try 16-bit (2 bytes per pixel)
    if file_size % 2 == 0:
        pixels_16 = file_size // 2
        print(f"16-bit interpretation: {pixels_16:,} pixels")
        
        # Find all factor pairs
        sqrt_val = int(np.sqrt(pixels_16))
        
        # Check for square dimensions first
        if sqrt_val * sqrt_val == pixels_16:
            print(f"   Perfect square: {sqrt_val} × {sqrt_val}")
            results.append(('16-bit', sqrt_val, sqrt_val))
        
        # Find other factors
        possible_widths = []
        for width in range(100, sqrt_val + 1):
            if pixels_16 % width == 0:
                height = pixels_16 // width
                # Prefer factors where width and height are close
                ratio = max(width, height) / min(width, height)
                if ratio < 3:  # Reasonable aspect ratio
                    possible_widths.append((width, height, ratio))
        
        if possible_widths:
            # Sort by aspect ratio (closer to 1.0 is better)
            possible_widths.sort(key=lambda x: x[2])
            print(f"  Possible dimensions (aspect ratio < 3):")
            for width, height, ratio in possible_widths[:10]:  # Show top 10
                aspect = f"{ratio:.2f}:1"
                print(f"    {width:5d} × {height:5d}  (aspect: {aspect})")
        
        print()
    
    # Try 8-bit (1 byte per pixel)
    pixels_8 = file_size
    print(f"8-bit interpretation: {pixels_8:,} pixels")
    
    sqrt_val = int(np.sqrt(pixels_8))
    
    if sqrt_val * sqrt_val == pixels_8:
        print(f"  ✓ Perfect square: {sqrt_val} × {sqrt_val}")
        results.append(('8-bit', sqrt_val, sqrt_val))
    
    possible_widths = []
    for width in range(100, sqrt_val + 1):
        if pixels_8 % width == 0:
            height = pixels_8 // width
            ratio = max(width, height) / min(width, height)
            if ratio < 3:
                possible_widths.append((width, height, ratio))
    
    if possible_widths:
        possible_widths.sort(key=lambda x: x[2])
        print(f"  Possible dimensions (aspect ratio < 3):")
        for width, height, ratio in possible_widths[:10]:
            aspect = f"{ratio:.2f}:1"
            print(f"    {width:5d} × {height:5d}  (aspect: {aspect})")
    
    print(f"\n{'='*70}")
    print("Recommendation: Check your XML metadata file for WIDTH/SAMPLE/LINE tags")
    print(f"{'='*70}\n")
    
    return results


# Run analysis on your IMG files
print("Analyzing IMG file dimensions\n")

for img_file in Path(INPUT_IMG_DIR).glob('*.img'):
    analyze_img_dimensions(str(img_file))


Analyzing IMG file dimensions...


Analyzing: m103947777lc.img
File size: 528,929,736 bytes

16-bit interpretation: 264,464,868 pixels

8-bit interpretation: 528,929,736 pixels
  Possible dimensions (aspect ratio < 3):
    16824 × 31439  (aspect: 1.87:1)

Recommendation: Check your XML metadata file for WIDTH/SAMPLE/LINE tags



In [ ]:
def extract_dimensions_from_xml(xml_filepath):
    """
    Extracts image dimensions from XML metadata file.
    
    Looks for common dimension tags in various NASA/ISRO formats.
    
    Args:
        xml_filepath: Path to XML metadata file
    
    Returns:
        Tuple (width, height) or None if not found
    """
    if not os.path.exists(xml_filepath):
        return None
    
    try:
        tree = ET.parse(xml_filepath)
        root = tree.getroot()
        
        # Common dimension tags in various formats
        dimension_patterns = {
            'width': ['WIDTH', 'SAMPLES', 'Sample_Width', 'IMAGE_WIDTH'],
            'height': ['HEIGHT', 'LINES', 'Sample_Height', 'IMAGE_HEIGHT'],
        }
        
        width = None
        height = None
        
        # Search for width
        for tag in dimension_patterns['width']:
            elem = root.find('.//' + tag)
            if elem is not None and elem.text:
                width = int(elem.text.strip())
                print(f"    Found WIDTH: {width} (tag: {tag})")
                break
        
        # Search for height
        for tag in dimension_patterns['height']:
            elem = root.find('.//' + tag)
            if elem is not None and elem.text:
                height = int(elem.text.strip())
                print(f"    Found HEIGHT: {height} (tag: {tag})")
                break
        
        if width and height:
            return (width, height)
        
    except Exception as e:
        pass
    
    return None


def extract_and_convert_with_xml(img_filepath):
    """
    Converts IMG file using dimensions from XML metadata.
    
    Args:
        img_filepath: Path to .img file
    
    Returns:
        Path to converted PNG or None
    """
    img_name = Path(img_filepath).stem
    xml_filepath = get_metadata_filepath(img_filepath)
    
    print(f"\n  Checking XML metadata: {os.path.basename(xml_filepath)}")
    
    dims = extract_dimensions_from_xml(xml_filepath)
    
    if not dims:
        print(f"  No dimensions found in XML")
        return None
    
    width, height = dims
    file_size = os.path.getsize(img_filepath)
    output_png = os.path.join(OUTPUT_PNG_DIR, f"{img_name}.png")
    
    # Try to convert with detected dimensions
    try:
        with open(img_filepath, 'rb') as f:
            raw_data = f.read()
        
        # Try 16-bit first (most common for lunar imagery)
        if width * height * 2 == file_size:
            print(f"  → Converting as {width}×{height} (16-bit)...")
            image_array = np.frombuffer(raw_data, dtype=np.uint16).reshape((height, width))
            normalized = (image_array.astype(np.float32) / 65535.0 * 255).astype(np.uint8)
            Image.fromarray(normalized).save(output_png)
            print(f"   Successfully converted to: {os.path.basename(output_png)}")
            return output_png
        
        # Try 8-bit
        elif width * height == file_size:
            print(f"   Converting as {width}×{height} (8-bit)...")
            image_array = np.frombuffer(raw_data, dtype=np.uint8).reshape((height, width))
            Image.fromarray(image_array).save(output_png)
            print(f"   Successfully converted to: {os.path.basename(output_png)}")
            return output_png
        
        else:
            print(f"   Dimension mismatch:")
            print(f"    Expected bytes: {width * height * 2} (16-bit) or {width * height} (8-bit)")
            print(f"    Actual bytes: {file_size}")
            return None
    
    except Exception as e:
        print(f"  ✗ Conversion failed: {e}")
        return None


# Test XML extraction on your files
print("\nExtracting dimensions from XML metadata files...\n")

for xml_file in Path(INPUT_METADATA_DIR).glob('*.xml'):
    img_name = xml_file.stem
    img_file = os.path.join(INPUT_IMG_DIR, f"{img_name}.img")
    
    if os.path.exists(img_file):
        print(f"Processing {img_name}:")
        dims = extract_dimensions_from_xml(str(xml_file))
        if dims:
            print(f"   Dimensions found: {dims[0]} × {dims[1]}")
        else:
            print(f"   No dimensions found in XML")
    else:
        print(f"Image file not found: {img_file}")



Extracting dimensions from XML metadata files...

Processing m103947777lc:
  ⚠ No dimensions found in XML
Image file not found: input_images\m103954939lc.img


## Step 9: Inspect Metadata

Utilities to inspect and verify generated metadata.

In [ ]:
def inspect_metadata_file(metadata_filepath):
    """
    Inspects and prints metadata file contents.
    
    Args:
        metadata_filepath: Path to metadata file (XML or JSON)
    """
    if not os.path.exists(metadata_filepath):
        print(f" File not found: {metadata_filepath}")
        return
    
    ext = Path(metadata_filepath).suffix.lower()
    
    if ext == '.xml':
        metadata = parse_xml_metadata(metadata_filepath)
    elif ext == '.json':
        with open(metadata_filepath, 'r') as f:
            metadata = json.load(f)
    else:
        print(f" Unsupported format: {ext}")
        return
    
    print(f"\nMetadata from: {os.path.basename(metadata_filepath)}")
    print("=" * 60)
    print(json.dumps(metadata, indent=2))


def compare_parent_child_metadata(parent_image_name):
    """
    Compares parent metadata with first child tile metadata.
    
    Args:
        parent_image_name: Name of parent image (without extension)
    """
    # Load parent metadata
    parent_xml = os.path.join(INPUT_METADATA_DIR, f"{parent_image_name}.xml")
    parent_metadata = parse_xml_metadata(parent_xml)
    
    # Find first child metadata
    child_pattern = f"{parent_image_name}_tile_*.{OUTPUT_METADATA_FORMAT}"
    child_files = list(Path(OUTPUT_METADATA_DIR).glob(child_pattern))
    
    if not child_files:
        print(f" No child metadata found for: {parent_image_name}")
        return
    
    child_file = str(child_files[0])
    
    print(f"\nCOMPARING METADATA")
    print("=" * 60)
    print(f"Parent: {os.path.basename(parent_xml)}")
    print(f"Child:  {os.path.basename(child_file)}")
    print("\nParent Metadata:")
    print("-" * 60)
    print(json.dumps(parent_metadata, indent=2))
    
    print("\nChild Metadata (with preprocessing info):")
    print("-" * 60)
    inspect_metadata_file(child_file)


def list_all_metadata_files():
    """
    Lists all generated metadata files.
    """
    metadata_files = list(Path(OUTPUT_METADATA_DIR).glob(f'*.{OUTPUT_METADATA_FORMAT}'))
    
    print(f"\nGenerated Metadata Files ({len(metadata_files)} total):")
    print("=" * 60)
    
    # Group by parent image
    from collections import defaultdict
    grouped = defaultdict(list)
    
    for f in metadata_files:
        # Extract parent name (everything before '_tile_')
        parent = f.stem.split('_tile_')[0] if '_tile_' in f.stem else f.stem
        grouped[parent].append(f.name)
    
    for parent, children in sorted(grouped.items()):
        print(f"\n{parent}: {len(children)} tiles")
        for child in sorted(children)[:5]:  # Show first 5
            print(f"  - {child}")
        if len(children) > 5:
            print(f"  ... and {len(children) - 5} more")


print(" Metadata inspection utilities defined")
print("\nFunctions available:")
print("  - inspect_metadata_file('path/to/file.json')")
print("  - compare_parent_child_metadata('image_name')")
print("  - list_all_metadata_files()")

✓ Metadata inspection utilities defined

Functions available:
  - inspect_metadata_file('path/to/file.json')
  - compare_parent_child_metadata('image_name')
  - list_all_metadata_files()


## Step 10: Example Usage

Examples of how to use the inspection functions.

In [71]:
# Example 1: List all generated metadata files
# list_all_metadata_files()

# Example 2: Inspect a specific metadata file
# inspect_metadata_file(os.path.join(OUTPUT_METADATA_DIR, 'your_tile_0000.json'))

# Example 3: Compare parent and child metadata
# compare_parent_child_metadata('your_image_name')


- two conversion methods:
  1. **GDAL Method** - Requires GDAL/OSGEO installation
     - Best for complex planetary image formats
     - Auto-detects image dimensions and bit depth
  2. **Binary Reading Method**  - No dependencies
     - Works for simple rectangular image files
     - Supports 8-bit and 16-bit grayscale
     - Tries common lunar image dimensions: 5000×5000, 2532×2304, 3000×3000, etc.

**Converting IMG files:**
```python
# Manually convert a single IMG file
from pathlib import Path
img_path = "input_images/your_image.img"
png_path = convert_img_to_png(img_path)  # Saves to output_png/
```


- **Tiles**: `{parent_name}_tile_{index}.png`
- **Masks**: `{parent_name}_tile_{index}_mask.png`
- **Metadata**: `{parent_name}_tile_{index}.{xml|json}`

### Preprocessing Information that are Added

Each tile metadata includes:
- **Tile Info**: ID, parent image, filenames (tile + mask)
- **Spatial Info**: Position (x,y), size, parent dimensions
- **Preprocessing**: Normalization method, percentiles, mean intensity
- **Shadow Detection**: Method, statistics, region count
- **Timestamp**: When the tile was processed

### Use

1. Place images in `input_images/` (PNG, JPG, or IMG format)
2. Place corresponding XML files in `input_metadata/`
3. Run `process_all_images()`
   - IMG files are automatically converted to PNG
   - All PNG files are then tiled and processed
4. Check outputs in respective directories
5. Use inspection functions to verify metadata



In [ ]:


required_dirs = [
    INPUT_IMG_DIR,
    INPUT_METADATA_DIR,
    OUTPUT_TILES_DIR,
    OUTPUT_MASKS_DIR,
    OUTPUT_METADATA_DIR,
    OUTPUT_VIZ_DIR
]

for d in required_dirs:
    os.makedirs(d, exist_ok=True)

print(" All required directories verified / created")


# ==============================
# 2️ VERIFY IMAGES EXIST
# ==============================

image_files = list(Path(INPUT_IMG_DIR).glob("*"))

if not image_files:
    print(f" No images found in {INPUT_IMG_DIR}")
    print("Please place your images inside the input directory before running.")
else:
    print(f" Found {len(image_files)} files in input directory")


# ==============================
# 3️ RUN BATCH PROCESSING
# ==============================

print("\nStarting full pipeline...\n")
summary = process_all_images()


# ==============================
# 4️ OPTIONAL: LIST GENERATED METADATA
# ==============================

print("\nListing generated metadata files:")
list_all_metadata_files()


#+============================================

generated_metadata = list(Path(OUTPUT_METADATA_DIR).glob(f"*.{OUTPUT_METADATA_FORMAT}"))

if generated_metadata:
    print("\nInspecting first metadata file:")
    inspect_metadata_file(str(generated_metadata[0]))
else:
    print("\nNo metadata files generated yet.")


✓ All required directories verified / created
✓ Found 1 files in input directory

Starting full pipeline...


BATCH PROCESSING: 2 images found

STEP 1: Converting IMG files to PNG

Converting: m103947777lc.img
  → Attempting XML extraction...
  → Converting with binary reading...


  File size: 528929736 bytes
  → Trying to find image dimensions from file size...
  ℹ File size: 528929736 bytes
  ℹ If 16-bit: 264464868 pixels
  ℹ If 8-bit: 528929736 pixels
  ✗ Could not auto-detect dimensions. Please specify manually or check XML metadata.

Converting: m103947777lc.img
  → Attempting XML extraction...
  → Converting with binary reading...
  File size: 528929736 bytes
  → Trying to find image dimensions from file size...
  ℹ File size: 528929736 bytes
  ℹ If 16-bit: 264464868 pixels
  ℹ If 8-bit: 528929736 pixels
  ✗ Could not auto-detect dimensions. Please specify manually or check XML metadata.

STEP 2: Processing 0 images

PROCESSING COMPLETE
✓ Successfully processed: 0/0 images
✓ Total tiles generated: 0
✓ Metadata format: JSON

⚠ Failed images (2):
  - m103947777lc.img
  - m103947777lc.img

Output locations:
  - Tiles: output_tiles/
  - Masks: output_masks/
  - Metadata: output_metadata/
  - Visualizations: output_visualizations/
  - Converted PNG: output_png/